# Extreme Weather Anomaly Tracking and Downscaling (Kaggle GPU Pipeline)

This notebook trains and evaluates the complete two-stage hybrid pipeline:
1. **Stage 1 (Tracker):** Spherical Icosahedral Multi-Mesh Graph Neural Network for detecting and tracking extreme anomalies across lead times (Day 3 to Day 10).
2. **Stage 2 (Downscaler):** CorrDiff-style conditional residual diffusion model downscaling from 12 km to 5 km while **preserving extreme disaster amplitudes** using physics-informed loss constraints.
3. **Alert Engine & Verification:** Hyper-local 5 km radius alerts and full probabilistic evaluation.

> **GitHub Repository:** [https://github.com/Aditri-web/weather-anomaly](https://github.com/Aditri-web/weather-anomaly)

### Step 1: Environment Verification (GPU & CUDA)

In [ ]:
import torch
import os
import sys

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.1f} MB")
    device = "cuda"
else:
    print("Running on CPU (Tip: turn on GPU accelerator in Kaggle settings: Settings -> Accelerator -> GPU P100/T4)")
    device = "cpu"

### Step 2: Clone Repository, Install Dependencies and Run Smoke Tests

In [ ]:
# If directory exists, pull latest changes; otherwise clone it
import os
if os.path.exists("/kaggle/working/weather-anomaly"):
    %cd /kaggle/working/weather-anomaly
    !git pull origin main
else:
    !git clone https://github.com/Aditri-web/weather-anomaly.git /kaggle/working/weather-anomaly
    %cd /kaggle/working/weather-anomaly

# Install requirements
!pip install -q pyyaml scipy pytest pydantic fastapi httpx uvicorn matplotlib

# Execute verification test suite (17 unit & integration tests)
!pytest tests/ -v

### Step 3: Train Stage 1 Spherical GNN Tracker

In [ ]:
!python scripts/train_tracker.py --config configs/tracker.yaml --epochs 5 --device {device}

### Step 4: Train Stage 2 CorrDiff Downscaler with Physics Constraints

In [ ]:
!python scripts/train_downscaler.py --config configs/downscaler.yaml --epochs 5 --device {device}

### Step 5: Run Full End-to-End Cycle Inference & Alert Generation

In [ ]:
!python scripts/infer.py --config configs/infer.yaml

### Step 6: Quantitative Performance Evaluation (Confidence, Precision, Recall, CSI, Peak Error, CRPS)

In [ ]:
# Run evaluation script to calculate Precision, Recall, F1, CSI, Peak Error, and CRPS
!python scripts/evaluate.py

### Step 7: Inspect Output Alerts & 5 km Core Impact Geometry

In [ ]:
import json

output_file = "data/outputs/cycles/cycle_2026-10-02T00Z.json"
with open(output_file, "r") as f:
    cycle_results = json.load(f)

print(f"Detected Tracks: {len(cycle_results['tracks'])}")
print(f"Generated Alerts: {cycle_results['count_alerts']}")
for alert in cycle_results['alerts']:
    print(f"\n[*] Alert: {alert['alert_id']} [{alert['category'].upper()}]")
    print(f"    Core Location: Lat {alert['core']['lat']}, Lon {alert['core']['lon']}")
    print(f"    Peak Intensity: {alert['peak_value']} mm/day | Exceedance Prob: {alert['exceedance_prob']:.1%}")
    print(f"    Lead Time: {alert['lead_time_hours']} hours ({alert['valid_from']} to {alert['valid_to']})")
    print(f"    Geodesic 5 km Polygon vertices: {len(alert['geometry']['coordinates'][0])} points")

### Step 8: Visualize Trajectory & 5 km Hyper-Local Warning Zone

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 8))

# Plot Bay of Bengal Region
ax.set_xlim(80.0, 95.0)
ax.set_ylim(10.0, 25.0)
ax.set_xlabel("Longitude (°E)", fontsize=12)
ax.set_ylabel("Latitude (°N)", fontsize=12)
ax.set_title("Cyclone Trajectory & Hyper-Local 5 km Alert Zones", fontsize=14, fontweight="bold")
ax.grid(True, linestyle="--", alpha=0.6)

# Plot tracks
for trk in cycle_results['tracks']:
    lats = [pt["lat"] for pt in trk["trajectory"]]
    lons = [pt["lon"] for pt in trk["trajectory"]]
    ax.plot(lons, lats, marker="o", linewidth=2.5, markersize=7, label=f"{trk['track_id']} Track (3-7 Days)")

# Plot 5 km alert circles
for alert in cycle_results['alerts']:
    coords = alert["geometry"]["coordinates"][0]
    poly_lons = [c[0] for c in coords]
    poly_lats = [c[1] for c in coords]
    color = "red" if alert["category"] == "severe" else ("orange" if alert["category"] == "moderate" else "yellow")
    ax.fill(poly_lons, poly_lats, color=color, alpha=0.35)
    ax.plot(poly_lons, poly_lats, color=color, linewidth=1.5)
    ax.scatter(alert["core"]["lon"], alert["core"]["lat"], color="black", s=20)

ax.legend(loc="upper left")
plt.tight_layout()
plt.show()